In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
fraud = pd.read_csv("../data/raw/Fraud_Data.csv")

In [2]:
fraud['signup_time'] = pd.to_datetime(fraud['signup_time'])
fraud['purchase_time'] = pd.to_datetime(fraud['purchase_time'])

In [3]:
fraud['time_since_signup'] = (
    fraud['purchase_time'] -
    fraud['signup_time']
).dt.total_seconds()

In [4]:
fraud['hour_of_day'] = fraud['purchase_time'].dt.hour

In [5]:
fraud['day_of_week'] = fraud['purchase_time'].dt.dayofweek

In [6]:
fraud['transaction_count'] = fraud.groupby('user_id')['user_id'].transform('count')

In [7]:
def ip_to_int(ip):
    return int(float(ip))

In [8]:
fraud['ip_int'] = fraud['ip_address'].apply(ip_to_int)

In [10]:
ip_df = pd.read_csv("../data/raw/IpAddress_to_Country.csv")

In [11]:
fraud = fraud.sort_values('ip_int')
ip_df = ip_df.sort_values('lower_bound_ip_address')

In [13]:
ip_df['lower_bound_ip_address'] = (
    ip_df['lower_bound_ip_address']
    .astype('int64')
)

ip_df['upper_bound_ip_address'] = (
    ip_df['upper_bound_ip_address']
    .astype('int64')
)

In [14]:
print(ip_df.dtypes)

lower_bound_ip_address    int64
upper_bound_ip_address    int64
country                     str
dtype: object


In [15]:
fraud = fraud.sort_values('ip_int')

ip_df = ip_df.sort_values(
    'lower_bound_ip_address'
)

In [16]:
fraud = pd.merge_asof(
    fraud,
    ip_df,
    left_on='ip_int',
    right_on='lower_bound_ip_address',
    direction='backward'
)

In [17]:
fraud = fraud[
    fraud['ip_int'] <= fraud['upper_bound_ip_address']
]

In [18]:
fraud.groupby('country')['class'].mean().sort_values(ascending=False).head(10)

country
Turkmenistan             1.000000
Namibia                  0.434783
Sri Lanka                0.419355
Luxembourg               0.388889
Virgin Islands (U.S.)    0.333333
Ecuador                  0.264151
Tunisia                  0.262712
Peru                     0.260504
Bolivia                  0.245283
Kuwait                   0.233333
Name: class, dtype: float64

In [19]:
fraud = pd.get_dummies(
    fraud,
    columns=['source','browser','sex'],
    drop_first=True
)

In [24]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

fraud[['purchase_value',
       'age',
       'time_since_signup']] = scaler.fit_transform(
    fraud[['purchase_value',
           'age',
           'time_since_signup']]
)

In [25]:
fraud.to_csv(
    "../data/processed/fraud_processed.csv",
    index=False
)